# Descriptive statistics and bivariate tests

This notebook reads the cleaned crash dataset produced by
[`loading_and_cleaning_data.ipynb`](loading_and_cleaning_data.ipynb), restricts the sample
to riders of micro-mobility vehicles (E-bike, Bike, E-PMD), and produces:

1. A summary-statistics table for one-hot-encoded categorical variables and a few
   continuous infrastructure variables.
2. A pivot table of counts and percentages by injury severity, together with
   bivariate tests (Chi-square for categorical variables; ANOVA or Kruskal-Wallis
   for continuous variables, depending on the normality test).

The two output tables are saved as CSV files and displayed inline at the end of
their respective sections so the analysis can be inspected directly in the notebook.


In [ ]:
# All imports for this notebook (kept together at the top, as required).
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, kruskal, kstest

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)


## 1. Load and filter the dataset

We keep only crashes involving micro-mobility vehicles (E-bike, Bike, E-PMD) and
drop rows where the severity is unknown (`severity == -1`).


In [ ]:
# low_memory=False removes the DtypeWarning caused by mixed-type columns in the
# combined CSV.
dataset = pd.read_csv('final_processed_crash_dataset_2.csv', low_memory=False)

dataset = dataset.loc[dataset['Vehicle'].isin(['E-bike', 'Bike', 'E-PMD'])]
dataset = dataset.loc[dataset['severity'] != -1]

# Convenience alias used in the bivariate tables further down.
dataset['sev'] = dataset['severity']

print(f'Rows after filtering: {len(dataset):,}')
print(f'Unique accidents:     {dataset["Num_Acc"].nunique():,}')
dataset.head()


## 2. Variable groups

We work with two units of observation:

- **Accident-level frame** (`data_acc`): one row per accident, used for variables
  that describe the crash environment (lighting, weather, road type, ...).
- **User-level frame** (`dataset`): one row per person involved, used for
  individual variables (age, gender, helmet, ...).

`var_for_acc` and `var_cont` list the variables that must be analysed at the
accident level and the continuous variables, respectively.


In [ ]:
data_acc = dataset.drop_duplicates('Num_Acc')

# Variables describing the crash itself (not the individuals involved).
var_for_acc = [
    'Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
    'int', 'atm', 'col', 'adr', 'lat', 'long', 'geometry', 'circ', 'nbv',
    'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ',
    'Weekend', 'Year', 'Time category', 'Lighting conditions',
    'Weather conditions', 'Involved vehicle 1 type', 'Point of impact',
    'Reglementation', 'Cycle facilities', 'Agglomeration', 'Max speed',
    'Road type', 'Intersection', 'Crossroad',
    'number of involved vehicles', 'Long profile', 'Truck traffic',
    'Pavement', 'Road width', 'Surface condition', 'Time of day'
]

# Continuous variables analysed in the bivariate section.
var_cont = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]


## 3. Summary statistics for one-hot-encoded categorical and continuous variables

We one-hot-encode a small set of categorical variables, merge in the continuous
infrastructure variables from the accident-level frame, and compute mean / median /
min / max for each column. The resulting table is saved as
`summary_statistics_categorical_continuous.csv` and is displayed below.


In [ ]:
var_added = pd.get_dummies(
    dataset[['Num_Acc', 'Cycle facilities', 'Pavement',
             'Crossroad', 'positionnement_piste',
             'Road type', 'Reglementation']]
)
var_added = var_added.astype(int)  # boolean -> integer

var_added = var_added.merge(
    data_acc[['Num_Acc', 'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit']],
    on='Num_Acc',
    how='left'
)

results_df = pd.DataFrame([
    {
        'Column': col,
        'Mean':   var_added[col].mean(),
        'Median': var_added[col].median(),
        'Min':    var_added[col].min(),
        'Max':    var_added[col].max(),
    }
    for col in var_added.columns
])

results_df.to_csv('summary_statistics_categorical_continuous.csv', index=False)
results_df


## 4. Bivariate analysis (counts, percentages, p-values)

For each categorical variable we report counts and percentages by severity level,
and a Chi-square p-value per category (NaN when expected frequencies are below 5,
to respect Cochran's rule). For each continuous variable we report the median and
interquartile range by severity, and an ANOVA p-value when the groups are normal
(Kolmogorov-Smirnov test) or a Kruskal-Wallis p-value otherwise.


In [ ]:
def calculate_chi2_p_value_for_each_category(data, categorical_var):
    """Per-category Chi-square p-value, NaN if Cochran's rule is violated."""
    p_values = {}
    for category in data[categorical_var].unique():
        contingency = pd.crosstab(data['severity'], data[categorical_var] == category)
        chi2, p, _, expected = chi2_contingency(contingency)
        if np.any(expected < 5):
            p_values[category] = np.nan
        else:
            p_values[category] = '<0.001' if p < 0.001 else str(round(p, 3))
    return p_values


def count_by_severity(data, categorical_var):
    """Counts and percentages of `categorical_var` within each severity level."""
    counts = (data
              .groupby(['severity', categorical_var])
              .size()
              .astype(int)
              .reset_index(name='count'))
    total_counts = counts.groupby(categorical_var)['count'].transform('sum')
    counts['percentage'] = (counts['count'] / total_counts * 100).round(2)
    counts['count_percentage'] = counts.apply(
        lambda row: f"{row['count']} ({row['percentage']}%)", axis=1
    )
    return counts


def compare_continuous_variable(data, group_var, continuous_var):
    """ANOVA if all groups are normal (KS test, alpha=0.05), Kruskal-Wallis otherwise."""
    groups_data = [
        data[data[group_var] == g][continuous_var].dropna().values
        for g in data[group_var].unique()
    ]
    normal = all(
        kstest(g, 'norm', args=(g.mean(), g.std())).pvalue > 0.05
        for g in groups_data
    )
    _, p_value = (f_oneway(*groups_data) if normal else kruskal(*groups_data))
    return '<0.001' if p_value < 0.001 else str(round(p_value, 3))


In [ ]:
categorical_vars = [
    'Age category', 'Gender', 'Vehicle', 'User category', 'Helmet', 'sev',
    'Point of impact_2', 'Reflective jacket', 'Lighting conditions',
    'Weather conditions', 'Point of impact', 'Maneuver', 'Cycle facilities',
    'Accident location', 'Trip purpose', 'Max speed', 'Intersection',
    'Crossroad', 'Long profile', 'Pavement', 'Surface condition', 'Road width',
    'vehicle_type_2', 'Maneuver_2', 'Gender_2', 'Age category involved',
    'positionnement_piste', 'Road type'
]

continuous_vars = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'largeurchaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]

pivot_results = []

# Categorical variables: counts/percentages and Chi-square per category.
for var in categorical_vars:
    df = data_acc if var in var_for_acc else dataset
    counts_df = count_by_severity(df, var)
    pivot_df = counts_df.pivot(index=var, columns='severity',
                               values='count_percentage').reset_index()
    pivot_df['Variable'] = var
    pivot_df['Category'] = pivot_df[var]
    pivot_df.drop(columns=var, inplace=True)
    p_values = calculate_chi2_p_value_for_each_category(df, var)
    pivot_df['p_value'] = pivot_df['Category'].map(p_values)
    pivot_results.append(pivot_df)

# Continuous variables: median [Q1-Q3] and ANOVA / Kruskal-Wallis p-value.
for var in continuous_vars:
    df = data_acc if var in var_for_acc else dataset
    df = df[['severity', var]].dropna(subset=[var])
    df = df[df[var] != 999]  # 999 is the Biogeme missing-data sentinel
    stats_df = df.groupby('severity')[var].agg(
        median='median',
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    ).reset_index()
    stats_df['median_iqr'] = stats_df.apply(
        lambda row: f"{row['median']:.2f} [{row['q1']:.2f}-{row['q3']:.2f}]", axis=1
    )
    stats_df = stats_df[['severity', 'median_iqr']].set_index('severity').T
    stats_df.columns = [1, 2, 3]
    stats_df['Variable'] = var
    stats_df['p_value'] = compare_continuous_variable(df, 'severity', var)
    pivot_results.append(stats_df)

final_pivot_df = pd.concat(pivot_results, ignore_index=True)
cols_order = ['Variable', 'Category'] + [c for c in final_pivot_df.columns
                                          if c not in ['Variable', 'Category']]
final_pivot_df = final_pivot_df[cols_order]

# Rename a few raw column names for the printed/exported table.
final_pivot_df = final_pivot_df.replace({
    'vehicle_type_2': 'Third-party vehicle type',
    'Maneuver_2': 'Third-party maneuver',
    'Gender_2': 'Third-party gender',
    'Point of impact_opposite': 'Third-party impact location',
    'vma': 'Speed limit',
    'surfacechaussee': 'Road surface width',
    'pentemoyenne': 'Average slope',
    'largeurtrottoirdroit': 'Sidewalk width',
    'age': 'Individual age',
    'number of involved vehicles': 'Number of vehicles involved',
})

print(f'Rows in the final pivot table: {len(final_pivot_df)}')
final_pivot_df


## 5. Export the bivariate table


In [ ]:
final_pivot_df.to_csv('descriptive_statistics_and_bivariate_tests.csv',
                      index=False, sep=';')
print('Saved to descriptive_statistics_and_bivariate_tests.csv')
final_pivot_df.head(20)
